In [2]:
import requests
import re
import pandas as pd
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

df = pd.read_csv(parent_dir+"\\02_Data\\raw\\gamefound_list.csv")

In [14]:
df

,name,projectID,creator,creatorID,homeUrl,currencySymbol,campaignGoal,fundsGathered,backersCount,campaignStart,...,playTime,playTimeDescription,playTimeUnit,pledgeManagerAvailability,fundedInSeconds,imageUrl,pledgeManagerSoftCloseDeadline,projectTags,enableShippingOnlyMode,originalType
0,Bloodstone,8454,Druid City Games,8843,https://gamefound.com/creators/druid-city-games,$,50000.0,95253.93,479,2026-06-02T13:00:00Z,...,60.0,NaN,0,NaN,7770,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Fantasy, Multiplayer, Competitive, Dice Game, ...",NaN,1
1,Aeolis,8619,Meeple Pug,3547,https://gamefound.com/creators/meeple-pug,€,NaN,33881.89,315,2026-06-02T15:00:00Z,...,60.0,NaN,0,NaN,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Area Control, Cooperative, Multiplayer, Campaign",NaN,2
2,A Life,9816,Skellig Games,2434,https://gamefound.com/creators/skellig-games,€,10000.0,22796.57,203,2026-06-02T15:00:00Z,...,60.0,NaN,0,NaN,3756,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Set Collection, Competitive, Family, Narrative...",NaN,1
3,Growing Season Deluxe Edition,8617,Undigital,5415,https://gamefound.com/creators/undigital,€,8000.0,15291.46,361,2026-06-02T16:00:00Z,...,30.0,NaN,0,NaN,4725,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Card Game, Strategy, Video Game Theme, Multipl...",NaN,1
4,Stonesaga Second Printing,9678,Open Owl Studios,474,https://gamefound.com/creators/oomm-games,$,25000.0,154784.30,1791,2026-05-26T15:00:00Z,...,90.0,NaN,0,NaN,1764,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Fantasy, Mythology, Cooperative, Campaign, Civ...",NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,Spring and Autumn: Story of China,1446,Mr. B. Games,1954,https://gamefound.com/creators/mr-b-games,$,30000.0,38009.00,219,2021-12-01T15:00:00Z,...,120.0,NaN,0,NaN,182152,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Exploration, History, Strategy, Area Control",NaN,1
2586,Crossroads Inn: The Board Game,1143,Klabater,1510,https://gamefound.com/creators/klabater,€,20000.0,13420.63,158,2021-10-28T14:00:00Z,...,120.0,NaN,0,NaN,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Fantasy, Strategy, Terrain Building, Humor",NaN,1
2587,PACHAMAMA,1170,SitDown,1514,https://gamefound.com/creators/sitdown,€,25000.0,16048.00,318,2021-10-19T17:00:00Z,...,60.0,NaN,0,NaN,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Exploration, Logical, Deduction, Terrain Building",NaN,1
2588,Taverns & Dragons (Canceled),1314,Lord Raccoon Games,356,https://gamefound.com/creators/thelastbottle,€,20000.0,20954.00,470,2021-10-12T13:00:00Z,...,60.0,NaN,0,NaN,433495,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Exploration, Card Game, Fantasy, Dice Game, St...",NaN,1


In [17]:
import requests
import re
import pandas as pd
import time
from tqdm.notebook import tqdm

project_urls = df['project_url']
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

rewards_list= []
fail_list=[]
for idx, row in tqdm(df.iterrows()):
    name = row['name']
    projectID_id = row['projectID']

    api_url = f"https://gamefound.com/api/projectContents/getRewards?projectID={projectID_id}"
    api_res = requests.get(api_url, headers=headers)
    if api_res.status_code == 200:
        data = api_res.json()
        if data['data']!=None:
            rewards_data = data.get('data').get('rewards')
        else:
            fail_list.append(api_url)            

        for item in rewards_data:
            item_dict = {
                'main_name' : name,
                'projectID_id' : projectID_id,
                "raw_url" : api_url,
                'anchorRelativeUrl': item.get('anchorRelativeUrl'),
                'backgroundUrl': item.get('backgroundUrl'),                # 상세페이지 대표 이미지
                'deliveryDateRemarks': item.get('deliveryDateRemarks'),
                'estimatedDeliveryAt': item.get('estimatedDeliveryAt'),    # 예상 배송일
                'hasDetails': item.get('hasDetails'),
                'isExposed': item.get('isExposed'),
                'isMostPopular': item.get('isMostPopular'),                # 가장 인기 있는 상품 여부
                'purchasedCopiesCount': item.get('purchasedCopiesCount'),  # 구매(후원)된 수량
                'additionalInfoUrl': item.get('additionalInfoUrl'),
                'hasInstallmentsAvailable': item.get('hasInstallmentsAvailable'), # 분할 납부 가능 여부
                'installmentCost': item.get('installmentCost'),
                'installmentMinPayment': item.get('installmentMinPayment'),
                'abstract': item.get('abstract'),                          # 상품 요약 설명
                'categoryID': item.get('categoryID'),
                'effectivePrice': item.get('effectivePrice'),              # 실제 결제 가격 (할인 적용 등)
                'hasLimitedStock': item.get('hasLimitedStock'),            # 한정 수량 여부
                'hasRetailerAccess': item.get('hasRetailerAccess'),
                'hasSpecialAccess': item.get('hasSpecialAccess'),
                'imageUrl': item.get('imageUrl'),                          # 상품 썸네일 이미지
                'isDigital': item.get('isDigital'),                        # 디지털 상품 여부
                'isDiscounted': item.get('isDiscounted'),                  # 할인 여부
                'isFeatured': item.get('isFeatured'),                      # 추천/강조 상품 여부
                'name': item.get('name'),                                  # 상품명
                'price': item.get('price'),                                # 원래 가격
                'productCanBePurchased': item.get('productCanBePurchased'), # 구매 가능 여부
                'productCanBePurchasedDescription': item.get('productCanBePurchasedDescription'),
                'productID': item.get('productID'),                        # 상품 고유 ID
                'projectID': item.get('projectID'),                        # 프로젝트 고유 ID
                'remainingStockLimit': item.get('remainingStockLimit'),    # 남은 수량
                'url': item.get('url'),                                    # 상품 상세 페이지 URL (상대 경로인 경우가 많음)
                'productState': item.get('productState')
            }
            rewards_list.append(item_dict)


    else:
        print(f"  {api_url} 오류 {api_res.status_code}")


time.sleep(1)

df1 = pd.DataFrame(rewards_list)

0it [00:00, ?it/s]

In [19]:
df1.to_csv(parent_dir+"\\02_Data\\raw\\rewards.csv", index=True, encoding="utf-8-sig")

In [ ]:
import requests
import re
import pandas as pd
import time
from tqdm.notebook import tqdm

project_urls = df['project_url']
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

rewards_list= []
fail_list=[]
for url in tqdm(project_urls):
    print(f"[{url}] 데이터 수집 중...")
    
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print(f"페이지 로드 실패: {res.status_code}")
        continue

    match = re.search(r'"projectID"\s*:\s*(\d+)', res.text, re.IGNORECASE)
    
    if match:
        projectID_id = int(match.group(1))
        
        api_url = f"https://gamefound.com/api/projectContents/getRewards?projectID={projectID_id}"
        api_res = requests.get(api_url, headers=headers)
        if api_res.status_code == 200:
            data = api_res.json()
            if data['data']!=None:
                rewards_data = data.get('data').get('rewards')
            else:
                fail_list.append(url)            

            for item in rewards_data:
                item_dict = {
                    "raw_url" : url+"\\rewards",
                    'anchorRelativeUrl': item.get('anchorRelativeUrl'),
                    'backgroundUrl': item.get('backgroundUrl'),                # 상세페이지 대표 이미지
                    'deliveryDateRemarks': item.get('deliveryDateRemarks'),
                    'estimatedDeliveryAt': item.get('estimatedDeliveryAt'),    # 예상 배송일
                    'hasDetails': item.get('hasDetails'),
                    'isExposed': item.get('isExposed'),
                    'isMostPopular': item.get('isMostPopular'),                # 가장 인기 있는 상품 여부
                    'purchasedCopiesCount': item.get('purchasedCopiesCount'),  # 구매(후원)된 수량
                    'additionalInfoUrl': item.get('additionalInfoUrl'),
                    'hasInstallmentsAvailable': item.get('hasInstallmentsAvailable'), # 분할 납부 가능 여부
                    'installmentCost': item.get('installmentCost'),
                    'installmentMinPayment': item.get('installmentMinPayment'),
                    'abstract': item.get('abstract'),                          # 상품 요약 설명
                    'categoryID': item.get('categoryID'),
                    'effectivePrice': item.get('effectivePrice'),              # 실제 결제 가격 (할인 적용 등)
                    'hasLimitedStock': item.get('hasLimitedStock'),            # 한정 수량 여부
                    'hasRetailerAccess': item.get('hasRetailerAccess'),
                    'hasSpecialAccess': item.get('hasSpecialAccess'),
                    'imageUrl': item.get('imageUrl'),                          # 상품 썸네일 이미지
                    'isDigital': item.get('isDigital'),                        # 디지털 상품 여부
                    'isDiscounted': item.get('isDiscounted'),                  # 할인 여부
                    'isFeatured': item.get('isFeatured'),                      # 추천/강조 상품 여부
                    'name': item.get('name'),                                  # 상품명
                    'price': item.get('price'),                                # 원래 가격
                    'productCanBePurchased': item.get('productCanBePurchased'), # 구매 가능 여부
                    'productCanBePurchasedDescription': item.get('productCanBePurchasedDescription'),
                    'productID': item.get('productID'),                        # 상품 고유 ID
                    'projectID': item.get('projectID'),                        # 프로젝트 고유 ID
                    'remainingStockLimit': item.get('remainingStockLimit'),    # 남은 수량
                    'url': item.get('url'),                                    # 상품 상세 페이지 URL (상대 경로인 경우가 많음)
                    'productState': item.get('productState')
                }
                rewards_list.append(item_dict)


        else:
            print(f"  {api_url} 오류 {api_res.status_code}")
            
    else:
        print(f"{url} projectID 검색 실패")

    time.sleep(1.5)

df1 = pd.DataFrame(rewards_list)

,raw_url,anchorRelativeUrl,backgroundUrl,deliveryDateRemarks,estimatedDeliveryAt,hasDetails,isExposed,isMostPopular,purchasedCopiesCount,additionalInfoUrl,...,isFeatured,name,price,productCanBePurchased,productCanBePurchasedDescription,productID,projectID,remainingStockLimit,url,productState
0,https://gamefound.com:443/en/projects/oomm-gam...,stonesaga-all-in-107951,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-01T00:00:00Z,False,False,False,211,None,...,True,Stonesaga All-In,359.0,True,None,107951,9678,NaN,/en/projects/oomm-games/stonesaga-second-print...,2
1,https://gamefound.com:443/en/projects/oomm-gam...,errata-pack-108768,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-18T00:00:00Z,False,False,False,1274,None,...,False,Errata Pack,7.5,True,None,108768,9678,NaN,/en/projects/oomm-games/stonesaga-second-print...,2
2,https://gamefound.com:443/en/projects/oomm-gam...,stonesaga-107938,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-01T00:00:00Z,False,False,False,78,None,...,False,Stonesaga,119.0,True,None,107938,9678,NaN,/en/projects/oomm-games/stonesaga-second-print...,2
3,https://gamefound.com:443/en/projects/oomm-gam...,stonesaga-expansions-107941,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-01T00:00:00Z,False,False,False,228,None,...,False,Stonesaga & Expansions,229.0,True,None,107941,9678,NaN,/en/projects/oomm-games/stonesaga-second-print...,2
4,https://gamefound.com:443/en/projects/studio-m...,new-content-reward-105788,https://imgcdn.gamefound.com/productimage/proj...,\n,2026-04-28T00:00:00Z,False,False,False,2459,None,...,True,New Content Reward,50.0,True,None,105788,8330,NaN,/en/projects/studio-midhall/beast-ashfall#/pro...,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11070,https://gamefound.com:443/en/projects/thelastb...,group-pledge-17696,https://imgcdn.gamefound.com/productimage/proj...,<div>This is an estimated delivery time.</div>,2022-09-01T00:00:00Z,False,False,False,0,None,...,False,Group Pledge,264.0,False,You can't add items to your pledge because thi...,17696,1314,NaN,/en/projects/thelastbottle/taverns-and-dragons...,2
11071,https://gamefound.com:443/en/projects/gindi/go...,champion-gameplay-all-in-18047,https://imgcdn.gamefound.com/productimage/proj...,None,2022-06-01T00:00:00Z,False,False,False,0,None,...,True,Champion (Gameplay All-in),72.0,False,You can't add items to your pledge because thi...,18047,1325,NaN,/en/projects/gindi/goldenstar-the-galactic-tou...,2
11072,https://gamefound.com:443/en/projects/gindi/go...,goldenstar-core-pledge-17700,https://imgcdn.gamefound.com/productimage/proj...,None,2022-06-01T00:00:00Z,False,False,False,0,None,...,False,Goldenstar Core Pledge,43.0,False,You can't add items to your pledge because thi...,17700,1325,NaN,/en/projects/gindi/goldenstar-the-galactic-tou...,2
11073,https://gamefound.com:443/en/projects/gindi/go...,challenger-18042,https://imgcdn.gamefound.com/productimage/proj...,None,2022-06-01T00:00:00Z,False,False,False,0,None,...,False,Challenger,69.0,False,You can't add items to your pledge because thi...,18042,1325,NaN,/en/projects/gindi/goldenstar-the-galactic-tou...,2


In [11]:
api_url

'https://gamefound.com/api/projectContents/getRewards?projectID=2401'

In [12]:
url

'https://gamefound.com:443/en/projects/le-sesame/vatrakill'

In [16]:
data

{'data': None, 'success': True, 'message': None}

In [21]:
data['data']!=None

False

In [16]:
df1

,main_name,projectID_id,raw_url,anchorRelativeUrl,backgroundUrl,deliveryDateRemarks,estimatedDeliveryAt,hasDetails,isExposed,isMostPopular,...,isFeatured,name,price,productCanBePurchased,productCanBePurchasedDescription,productID,projectID,remainingStockLimit,url,productState
0,Bloodstone,8454,https://gamefound.com/api/projectContents/getR...,gameplay-all-in-deluxe-pledge-107665,https://imgcdn.gamefound.com/productimage/proj...,None,2027-05-30T00:00:00Z,False,False,False,...,True,Gameplay All-In Deluxe Pledge,230.0,True,None,107665,8454,NaN,/en/projects/druid-city-games/bloodstone#/prod...,2
1,Bloodstone,8454,https://gamefound.com/api/projectContents/getR...,retail-base-edition--108009,https://imgcdn.gamefound.com/productimage/proj...,None,2025-06-01T00:00:00Z,False,False,False,...,False,Retail Base Edition,75.0,True,None,108009,8454,NaN,/en/projects/druid-city-games/bloodstone#/prod...,2
2,Bloodstone,8454,https://gamefound.com/api/projectContents/getR...,deluxe-base-edition-107663,https://imgcdn.gamefound.com/productimage/proj...,None,2027-05-30T00:00:00Z,False,False,False,...,False,Deluxe Base Edition,140.0,True,None,107663,8454,NaN,/en/projects/druid-city-games/bloodstone#/prod...,2
3,Aeolis,8619,https://gamefound.com/api/projectContents/getR...,aeolis-with-metal-coins-english-version--107326,https://imgcdn.gamefound.com/productimage/proj...,None,2026-06-09T00:00:00Z,False,False,False,...,True,Aeolis with Metal Coins - English Version 🇬🇧,199.0,True,None,107326,8619,6.0,/en/projects/meeple-pug/aeolis#/product/107326,2
4,Aeolis,8619,https://gamefound.com/api/projectContents/getR...,aeolis-english-version--107324,https://imgcdn.gamefound.com/productimage/proj...,None,2026-06-09T00:00:00Z,False,False,False,...,False,Aeolis - English Version 🇬🇧,169.0,True,None,107324,8619,0.0,/en/projects/meeple-pug/aeolis#/product/107324,2
5,Aeolis,8619,https://gamefound.com/api/projectContents/getR...,aeolis-german-version--107332,https://imgcdn.gamefound.com/productimage/proj...,None,2026-06-09T00:00:00Z,False,False,False,...,False,Aeolis - German Version 🇩🇪,169.0,True,None,107332,8619,0.0,/en/projects/meeple-pug/aeolis#/product/107332,2
6,Aeolis,8619,https://gamefound.com/api/projectContents/getR...,aeolis-french-version--107334,https://imgcdn.gamefound.com/productimage/proj...,None,2026-06-09T00:00:00Z,False,False,False,...,False,Aeolis - French Version 🇫🇷,169.0,True,None,107334,8619,0.0,/en/projects/meeple-pug/aeolis#/product/107334,2
7,Aeolis,8619,https://gamefound.com/api/projectContents/getR...,aeolis-with-metal-coins-german-version--107333,https://imgcdn.gamefound.com/productimage/proj...,None,2026-06-09T00:00:00Z,False,False,False,...,False,Aeolis with Metal Coins - German Version 🇩🇪,199.0,True,None,107333,8619,0.0,/en/projects/meeple-pug/aeolis#/product/107333,2
8,Aeolis,8619,https://gamefound.com/api/projectContents/getR...,aeolis-with-metal-coins-french-version--107335,https://imgcdn.gamefound.com/productimage/proj...,None,2026-06-09T00:00:00Z,False,False,False,...,False,Aeolis with Metal Coins - French Version 🇫🇷,199.0,True,None,107335,8619,2.0,/en/projects/meeple-pug/aeolis#/product/107335,2
9,A Life,9816,https://gamefound.com/api/projectContents/getR...,a-life-deluxe-gamefound-exclusive-108401,https://imgcdn.gamefound.com/productimage/proj...,None,None,False,False,False,...,True,A Life Deluxe - Gamefound Exclusive,100.0,True,None,108401,9816,NaN,/en/projects/skellig-games/a-life#/product/108401,2
